# 4-6 字体设置

Matplotlib 默认字体不一定包含中文字形，因此中文标题、坐标轴标签或图例可能显示为方框。
解决方法是选择确实包含中文字形的字体。

本节选择可用的中文字体，学习两种方法：

1. 通过 `FontProperties` 给指定文字设置字体
2. 注册字体后，通过运行时参数设置默认字体

> 优先使用项目中的可选字体文件或系统现有字体，不修改 Matplotlib 的全局配置文件。
> 没有可用的中文字体时会显示提示，绘图仍可运行，但中文可能显示为方框。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties

## 1. 准备绘图数据

创建直线 `y = 2x + 5`。`np.arange(-5, 11)` 包含 `-5` 到 `10`，不包含结束值 `11`。
这里包含负数，后面可以同时检查负号是否正常显示。

In [ ]:
x = np.arange(-5, 11)
y = 2 * x + 5

plt.figure(figsize=(7, 4))
plt.plot(x, y)
plt.title('Example: y = 2x + 5')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()

英文通常能直接显示。将标题改成中文后是否能显示，取决于当前配置的字体，
不能仅凭一台机器上的效果判断其他环境也能正常显示。

## 2. 选择字体并找到字体文件

`Path('.')` 表示内核的当前工作目录，`Path('..')` 表示上一级目录。
下面支持从项目根目录或本章目录运行。
程序先检查项目相对路径 `fonts/NotoSansCJKsc-Regular.otf` 和 `fonts/NotoSansSC-Regular.ttf`；
这两个文件是可选的，没有时会选择系统已安装的中文字体。

`font_manager.findfont()` 根据字体名称找到文件路径，因此不需要写死某台电脑的安装目录。
如果候选中文字体都不可用，就使用 Matplotlib 自带的 `DejaVu Sans` 并显示提示。
该字体不包含完整中文字形，需要安装中文字体或提供上述可选文件，才能正常显示中文。

In [ ]:
# fonts 文件夹是可选的；没有字体文件时，使用系统已安装的字体。
project_dir = Path('.')
if Path.cwd().name == '第4章-Matplotlib':
    project_dir = Path('..')
font_files = [
    project_dir / 'fonts' / 'NotoSansCJKsc-Regular.otf',
    project_dir / 'fonts' / 'NotoSansSC-Regular.ttf',
]
font_path = next((path for path in font_files if path.is_file()), None)

if font_path is None:
    font_candidates = [
        'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
        'Source Han Sans SC', 'PingFang SC', 'Heiti SC', 'Songti SC',
        'Arial Unicode MS',
    ]
    available_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    font_name = next((name for name in font_candidates if name in available_fonts), None)
    if font_name is None:
        font_name = 'DejaVu Sans'
        print('未找到中文字体，暂用 DejaVu Sans，图中的中文可能显示为方框。')
        print('可安装 Noto Sans CJK SC，或将中文字体文件放入项目的 fonts 文件夹后重新运行。')
    font_path = Path(font_manager.findfont(
        font_manager.FontProperties(family=font_name), fallback_to_default=False
    ))
else:
    font_name = font_manager.FontProperties(fname=str(font_path)).get_name()

print('选择的字体：', font_name)

## 3. 方法一：给指定文字设置字体

`FontProperties(fname=..., size=...)` 根据字体文件创建字体属性对象。

| 参数 | 作用 |
|---|---|
| `fname` | 字体文件路径 |
| `size` | 字体大小，单位为磅 |
| `fontproperties` | 把字体属性传给标题或坐标轴文字 |
| 图例的 `prop` | 为图例设置字体属性 |

这种方法只影响明确指定了字体的文字。中文刻度标签等其他文字也需要单独指定。

In [ ]:
font = FontProperties(fname=str(font_path), size=13)

plt.figure(figsize=(7, 4))
plt.plot(x, y, label='直线')
plt.title('使用字体文件显示中文', fontproperties=font)
plt.xlabel('x 轴', fontproperties=font)
plt.ylabel('y 轴', fontproperties=font)
plt.legend(prop=font)
plt.tight_layout()
plt.show()

## 4. 查看可用字体

`font_manager.fontManager.ttflist` 保存当前字体管理器发现的字体条目。
使用集合去除重复名称，再排序，便于查找。

Windows 上常见 `Microsoft YaHei` 或 `SimHei`，macOS 上可能有 `PingFang SC` 或 `Songti SC`，
Linux 上可以使用 `Noto Sans CJK SC`。实际可用字体取决于当前环境。

In [ ]:
font_names = sorted({entry.name for entry in font_manager.fontManager.ttflist})
print('发现的字体名称数量：', len(font_names))
print('前 15 个字体名称：', font_names[:15])

## 5. 方法二：设置运行时默认字体

### 5.1 注册字体文件

`addfont()` 把字体文件注册到当前 Python 进程，随后可以通过字体名称引用它。
系统已发现的字体通常不需要再次注册，这里统一演示文件注册的方法，便于使用项目中的可选字体。
这种注册不会安装系统字体，也不会永久保留；重启内核后需要重新运行。

In [ ]:
font_manager.fontManager.addfont(str(font_path))
font_name = FontProperties(fname=str(font_path)).get_name()
print('本次使用的字体名称：', font_name)

### 5.2 用 `rc_context()` 临时应用字体设置

| 设置 | 含义 |
|---|---|
| `'font.family': [font_name]` | 指定默认字体 |
| `'axes.unicode_minus': False` | 让负刻度使用普通连字符，兼容缺少 Unicode 负号字形的字体 |

`plt.rc_context()` 只在 `with` 代码块中应用这些设置，退出后自动恢复。
中文标题、坐标轴标签和图例都会使用这里的默认字体。

In [ ]:
with plt.rc_context({'font.family': [font_name], 'axes.unicode_minus': False}):
    plt.figure(figsize=(7, 4))
    plt.plot(x, y, marker='o', label='直线 y = 2x + 5')
    plt.title('使用默认字体显示中文', fontsize=18)
    plt.xlabel('x 轴', fontsize=13)
    plt.ylabel('y 轴', fontsize=13)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### 5.3 在本次内核会话中持续使用

如果整本笔记都要使用中文，可以直接修改 `plt.rcParams`。
后续新创建的图表会继续采用这些设置，直到再次修改或重启内核。
这只是运行时设置，不是修改磁盘上的 `matplotlibrc` 配置文件。

In [ ]:
plt.rcParams['font.family'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False

## 6. 常见问题

| 现象 | 检查方法 |
|---|---|
| 提示未找到中文字体 | 安装候选中文字体，或在项目 `fonts` 文件夹中放入第 2 部分列出的可选文件 |
| 中文显示为方框 | 确认所选字体包含中文字形，并应用到了对应文字 |
| 仅负号显示异常 | 检查 `axes.unicode_minus` 设置 |
| 重启后字体名称找不到 | 重新运行字体查找、`addfont()` 和字体设置单元格 |
| 出现缺字警告 | 检查字体及具体缺失字符，不要直接屏蔽全部警告 |

屏蔽警告不能补全缺失的字形，应选择包含所需字符的字体。

## 7. 要点总结

- 中文能否显示取决于字体是否包含相应字形。
- `FontProperties(fname=...)` 可以直接使用字体文件，为特定文字指定字体。
- `font_manager` 可以查看字体信息，`findfont()` 查找字体文件，`addfont()` 临时注册字体。
- `rc_context()` 适合局部配置，`rcParams` 适合本次内核会话持续使用。
- `fontsize` 或 `FontProperties(size=...)` 控制文字大小。
- 项目字体使用相对路径，系统字体使用 `findfont()` 查找，无须填写个人电脑的安装路径。